In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import joblib
import shap 

plt.rcParams['figure.figsize'] = (9, 6)
plt.rcParams['font.size'] = 12
sns.set_theme(style="whitegrid")

In [ ]:
model = joblib.load('models/best_model.pkl')
scaler = joblib.load('models/scaler.pkl')
le = joblib.load('models/label_encoder.pkl')

df = pd.read_csv('data/processed/spotify_clean.csv')
X = df.drop('track_genre', axis=1)
y_raw = df['track_genre']
y = le.transform(y_raw)
class_names = le.classes_

X_scaled = scaler.transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

print(f"Загружено: {len(X_scaled_df)} треков, {len(class_names)} жанров")
print(f"Модель: {model.__class__.__name__}")

In [ ]:
explainer = shap.TreeExplainer(model)

shap_values = explainer.shap_values(X_scaled_df)

print('Строим глобальный график важности:')
shap.summary_plot(shap_values, X_scaled_df, plot_type='bar', show = False)
plt.title('Глобальная важность признаков')
plt.xlabel('Средняя абсолютная велчина SHAP')
plt.tight_layout()
plt.show()

In [ ]:
# локальное объяснение
idx = 10
row = X_scaled_df.iloc[idx:idx+1]
pred_clases = model.predict(row)[0]
pred_genre = le.inverse_transform([pred_clases]) 

print(f"Трек #{idx} | Предсказание модели: {pred_genre[0]}")

shap_vals_for_class = shap_values[pred_clases][idx] 
shap.plots.waterfall(
    shap.Explanation(
        values = shap_vals_for_class,
        base_values = explainer.expected_value[pred_clases],  
        data = row.iloc[0],
        feature_names = row.columns
    ),
    max_display = 10
)